In [2]:
import yfinance as yf
import pandas as pd
from dotenv import load_dotenv
import os
import requests
load_dotenv()

DB_CONFIG = {
    'dbname': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host': 'localhost',
    'port': 5432
}

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

response = requests.get('https://en.wikipedia.org/wiki/FTSE_100_Index', headers=headers)

ftse = pd.read_html(response.text)[6]
ftse_tickers = ftse['Ticker']
ftse_tickers = ftse_tickers.to_list()
ftse_tickers = [ticker + '.L' for ticker in ftse_tickers]

response = requests.get('https://en.wikipedia.org/wiki/S%26P_100', headers=headers)
snp = pd.read_html(response.text)[2]
snp_tickers = snp['Symbol']
snp_tickers = snp_tickers.to_list()
snp_tickers[snp_tickers.index('BRK.B')] = 'BRK-B'

tickers = yf.Tickers(' '.join(ftse_tickers + snp_tickers))

ticker_df = pd.DataFrame(columns=['ticker', 'sector'])
ticker_df.ticker = ftse_tickers + snp_tickers

sectors = []
for ticker in tickers.tickers.values():
    try:
        sectors.append(ticker.info['sector'])
    except KeyError:
        sectors.append(None)

ticker_df.sector = sectors

import psycopg as pg

with pg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        for _, row in ticker_df.iterrows():
            cur.execute(f"INSERT INTO tickers (ticker, sector) VALUES ('{row['ticker']}', '{row['sector']}');")
            
        conn.commit()

/var/folders/78/5rn_2_c91lg2f4yy_2jl7j2w0000gn/T/ipykernel_51507/3359012706.py:22: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  ftse = pd.read_html(response.text)[6]
/var/folders/78/5rn_2_c91lg2f4yy_2jl7j2w0000gn/T/ipykernel_51507/3359012706.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  snp = pd.read_html(response.text)[2]


ValueError: 

In [9]:
prices2 = yf.download(snp_tickers + ftse_tickers, start='2005-01-01', end='2009-12-31', auto_adjust=False)

[*********************100%***********************]  200 of 201 completed

30 Failed downloads:
['META', 'NOW', 'TSLA', 'PLTR', 'UBER', 'PYPL', 'GM', 'ABBV']: YFPricesMissingError('possibly delisted; no price data found  (1d 2005-01-01 -> 2009-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1104555600, endDate = 1262235600")')
['MSFT']: Timeout('Failed to perform, curl: (28) Operation timed out after 10002 milliseconds with 32085 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['AAF.L', 'PSH.L', 'CCH.L', 'CTEC.L', 'CCEP.L', 'AUTO.L', 'GLEN.L', 'HLN.L', 'EDV.L', 'MNG.L', 'MTLN.L']: YFPricesMissingError('possibly delisted; no price data found  (1d 2005-01-01 -> 2009-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1104537600, endDate = 1262217600")')
['MET']: Timeout('Failed to perform, curl: (28) Operation timed out after 10001 milliseconds with 20795 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first f

In [10]:
prices = prices2.stack(level="Ticker").reset_index()

/var/folders/78/5rn_2_c91lg2f4yy_2jl7j2w0000gn/T/ipykernel_51507/2048080462.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  prices = prices2.stack(level="Ticker").reset_index()


In [ ]:
import psycopg as pg
with pg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        # cur.execute(f"SELECT COUNT(*) FROM daily")
        # print(cur.fetchall())
        for _, row in prices.iterrows():
            cur.execute(f"INSERT INTO daily (ticker, date, open, high, low, close, volume) VALUES ('{row['Ticker']}', '{row['Date']}', '{row['Open']}', '{row['High']}', '{row['Low']}', '{row['Close']}', '{int(row['Volume'])}');")

        conn.commit()

[(980209,)]


In [23]:
prices[(prices["Ticker"]=="ENT.L") & (prices["Date"]=="2015-02-24")]

Price,Date,Ticker,Adj Close,Close,High,Low,Open,Volume
6858,2015-02-24,ENT.L,480.602325,482.0,484.0,474.899994,474.5,330977.0
